## 2일차 목차

| 번호 | 주제 | 핵심 내용 |
|---|---|---|
| 1 | 빠른 복습: 기본 RAG 흐름 | 기본 RAG 흐름 |
| 2 | Advanced RAG 기법 | Multi-Query, HyDE, Hybrid, 후보군 정리, Generation |
| 3 | RAG 성능 평가 (Evaluation) | Hit@K, Recall@K, Faithfulness |
| 4 | RAG 운영 및 모니터링 (Monitoring & Observability) | Observability, Langfuse, trace |
| 5 | 최종 과제 안내 | 과제 흐름, 제출물, 채점 기준 |
| 6 | 참고자료 | 대표 논문과 도구 참고 |


## 1. 빠른 복습: 기본 RAG 흐름

<img src="image/RAG.png" width="640">

2일차의 Advanced RAG는 이 기본 흐름을 버리는 것이 아니라,  
각 단계(Query, Retrieval, Post-Retrieval, Generation, Evaluation)를 더 정교하게 다듬는 과정입니다.


## 2. Advanced RAG 기법

1일차에서 본 기본 RAG는 `검색 -> 답변 생성` 흐름을 이해하는 출발점이었습니다.  
2일차에서는 그 흐름을 버리는 것이 아니라, 각 단계를 더 정교하게 다듬는 방법을 봅니다.

오늘 볼 관점은 아래 다섯 가지입니다.
- **Pre-Retrieval**: 검색 전에 질문이나 입력을 더 잘 찾히는 형태로 바꾸는 단계
- **Retrieval**: 정답 가능성이 있는 문서를 최대한 빠뜨리지 않고 모으는 단계
- **Post-Retrieval**: 모아온 후보를 다시 읽고, 중복·잡음·불필요한 길이를 줄이는 단계
- **Generation**: 근거 기반 답변과 출처 표시
- **Evaluation / Monitoring**: 개선 결과를 확인하기

정리하면,
- `Pre-Retrieval`은 **찾기 전에 질문을 다듬는 단계**이고
- `Post-Retrieval`은 **찾은 뒤 문서를 다듬는 단계**입니다.

핵심은 **어떤 기법이 어느 단계의 문제를 줄이는지** 흐름으로 보는 것입니다.

<img src="image/paradigms_of_RAG.png" width="600">



### 2.1 Query Transformation 확장: Multi-Query Retrieval

이 구간은 **Pre-Retrieval**에 해당합니다.  
즉, 검색기를 바꾸기 전에 **질문 자체를 더 잘 찾히는 형태로 바꾸는 단계**입니다.

1일차에서는 하나의 질문을 더 또렷하게 바꾸는 **Query Rewriting**을 살펴봤습니다.  
여기서는 그 다음 단계로, 하나의 질문을 **여러 검색 질문으로 확장**하는 방법을 봅니다.

**Multi-Query Retrieval**은 하나의 질문에서 여러 검색 쿼리를 만든 뒤,  
각 쿼리를 독립적으로 검색하고 결과를 합쳐 중복을 정리하는 방식입니다.

흐름은 단순합니다.

1. **LLM이 원 질문을 여러 검색 질문으로 생성**
2. **각 쿼리를 Retriever에 독립적으로 질의**
3. **결과를 합치고 같은 문서는 한 번만 남김**

이 방식은 결과가 항상 더 좋아진다기보다,  
**원 질문 하나로는 놓칠 수 있는 관련 후보를 더 넓게 모으는 데** 의미가 있습니다.


##### 실습: 질문 생성과 검색 결과 비교

여기서는 **원 질문 상위 결과**와 **확장 질문을 합친 뒤의 상위 결과**를 비교합니다.  
핵심은 정답을 바로 맞히는지보다, **관련 후보를 더 넓게 모으는지** 보는 것입니다.


In [ ]:
import os  # 파일 경로와 환경 변수를 다룰 때 쓰는 표준 라이브러리입니다.
import warnings  # 실습과 무관한 경고를 숨길 때 쓰는 표준 라이브러리입니다.

warnings.filterwarnings("ignore")  # 실습과 무관한 경고는 모두 숨깁니다.

from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션에 불러옵니다.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 함께 불러옵니다.
from langchain_community.vectorstores import FAISS  # 임베딩 벡터를 저장하고 유사 문서를 찾는 벡터 DB입니다.
from langchain_community.document_loaders import PyPDFLoader  # PDF를 페이지 단위 문서로 읽어옵니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥을 최대한 살리며 긴 문서를 여러 청크로 나눕니다.
from pdf_utils import clean_pdf_documents  # PDF 정제 공통 유틸을 불러옵니다.

load_dotenv(override=True)  # 실습에 필요한 API 키를 현재 세션에 반영합니다.

PDF_PATH = "data/금융투자협회_투자길라잡이_2018.pdf"  # 2일차 실습의 공통 원본 문서입니다.
DB_PATH = "./faiss_index_kofia_guide_cleaned"  # 정제한 문서를 저장한 벡터 DB 경로입니다.

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 문서와 질문을 같은 벡터 공간으로 바꿉니다.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# 2일차에서는 Hybrid와 Evaluation이 같은 청킹 기준을 봐야 하므로 split_docs를 항상 준비합니다.
loader = PyPDFLoader(PDF_PATH)
# clean_pdf_documents()는 머리말/쪽번호/깨진 글리프를 정리해 이후 청킹 기준을 안정화합니다.
docs = clean_pdf_documents(loader.load())
# 문단 -> 문장 -> 단어 순서로 경계를 시도합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", ".", " ", ""],
)
split_docs = text_splitter.split_documents(docs)
for idx, d in enumerate(split_docs):
    d.metadata["id"] = idx  # Dense/BM25/Evaluation이 같은 청크 ID를 공유하게 맞춥니다.

# 이미 저장된 벡터 DB가 있으면 재사용하고, 없으면 같은 split_docs로 새로 만듭니다.
if os.path.exists(DB_PATH):
    vectorstore = FAISS.load_local(
        folder_path=DB_PATH,
        embeddings=embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    vectorstore.save_local(DB_PATH)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 평가의 Recall@5까지 무리 없이 보도록 여유 있게 가져옵니다.
print("공통 실습 준비 완료")
print("정제된 청크 수:", len(split_docs))


In [ ]:
import logging  # 실행 로그의 출력 수준을 조절할 때 쓰는 표준 라이브러리입니다.
from typing import List  # 함수 입력과 출력 타입을 읽기 쉽게 적을 때 씁니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼내는 파서입니다.
from langchain_core.documents import Document  # 텍스트와 메타데이터를 함께 담는 문서 객체입니다.
from langchain_core.runnables import RunnableLambda  # 일반 파이썬 함수를 체인 단계로 연결합니다.

# 한 질문을 서로 다른 검색 표현 2개로 넓히는 프롬프트입니다.
multi_query_prompt = PromptTemplate.from_template("""
당신은 금융투자협회의 투자자 안내서에서 정보를 찾는 검색 봇입니다.
사용자의 질문을 바탕으로, 실제 문서 본문에 등장할 법한 **구체적인 검색 쿼리 2개**를 생성하세요.

[작성 가이드]
1. 질문에 답하지 말고 **검색창에 넣을 문장형 쿼리**만 출력할 것
2. 두 질문은 같은 주제를 유지하되, 표현이나 초점을 조금씩 다르게 할 것
3. 문서에 없는 다른 주제로 퍼지지 말 것
4. 짧은 구어체보다 문서 본문에 가까운 표현으로 바꿀 것
5. 결과는 불릿 2개로만 출력할 것

원본 질문: {question}

출력 예시:
- 고객 동의 없이 직원이 매매한 경우 어떤 분쟁으로 보는가?
- 위탁받지 않고 매매한 경우 임의매매로 판단하는 기준은 무엇인가?
""")

# 모델이 만든 여러 줄 문자열을 실제 검색 질의 리스트로 바꿉니다.
def parse_and_log_queries(llm_output: str) -> List[str]:  # 모델 출력을 검색용 질문 리스트로 바꿉니다.
    queries = [line.strip("- ").strip() for line in llm_output.split("\n") if line.strip()]  # 불릿과 빈 줄을 제거합니다.
    print("\n[AI가 생성한 확장 질문들]")
    for i, q in enumerate(queries, 1):
        print(f"{i}. {q}")
    return queries

# 각 쿼리 결과를 순서대로 합치고, 같은 문서는 한 번만 남깁니다.
def merge_multi_query_results(result_lists: List[List[Document]]) -> List[Document]:  # 여러 질문 결과를 하나로 합칩니다.
    merged_docs = []  # 최종 비교에 보여 줄 문서를 모읍니다.
    seen_ids = set()  # 이미 본 문서는 중복 추가하지 않기 위해 기록합니다.

    for docs in result_lists:
        for doc in docs:
            doc_id = doc.metadata.get("id", doc.page_content)  # 문서마다 비교 기준이 되는 ID를 꺼냅니다.
            if doc_id in seen_ids:
                continue
            seen_ids.add(doc_id)
            merged_docs.append(doc)

    return merged_docs

# 검색 결과를 짧게 비교 출력합니다.
def preview_docs(label: str, docs: List[Document], max_items: int = 3):  # 검색 결과를 짧게 미리 봅니다.
    print(f"\n=== {label} ===")
    for i, doc in enumerate(docs[:max_items], 1):
        page = doc.metadata.get("page")
        page_text = f", page={page + 1}" if isinstance(page, int) else ""
        print(f"[{i}] id={doc.metadata.get('id')}{page_text}")
        print(doc.page_content[:180])  # 본문 앞부분만 잘라서 비교합니다.
        print("---")

# 프롬프트 → 모델 → 문자열 → 질의 리스트 흐름을 하나의 체인으로 묶습니다.
multi_query_chain = (
    multi_query_prompt  # 질문을 확장하는 프롬프트를 만듭니다.
    | llm  # 프롬프트를 바탕으로 새 검색 질문을 생성합니다.
    | StrOutputParser()  # 모델 응답을 문자열로 꺼냅니다.
    | RunnableLambda(parse_and_log_queries)  # 문자열을 질문 리스트로 바꿉니다.
)

# 원 질문은 일부러 구어체로 두고, 확장 질문이 후보를 넓히는지 봅니다.
question = "고객 동의 없이 직원이 알아서 매매하면 어떤 분쟁이 되는가?"
queries = multi_query_chain.invoke({"question": question})  # 원 질문에서 확장 질문 2개를 만듭니다.

In [ ]:
# 원 질문 검색과 확장 질문 검색 결과를 나란히 비교합니다.
original_docs = retriever.invoke(question)  # 원 질문만 넣었을 때의 결과입니다.
expanded_result_lists = [retriever.invoke(q)[:2] for q in queries]  # 확장 질문마다 상위 2개씩 가져옵니다.
merged_docs = merge_multi_query_results(expanded_result_lists)  # 여러 질문 결과를 중복 없이 합칩니다.

preview_docs("원 질문 검색 결과", original_docs, max_items=2)
preview_docs("확장 질문 병합 결과", merged_docs, max_items=3)

지금까지는 **질문 자체를 더 잘 만들어 검색 품질을 높이는 방법**을 살펴봤습니다.  
이제는 질문 대신 **가상의 답변을 만들어 검색하는 HyDE**와, **서로 다른 검색기를 조합하는 Hybrid Retrieval**로 넘어갑니다.



### 2.2 HyDE (Hypothetical Document Embedding)

HyDE는 질문을 바로 검색에 사용하지 않고, LLM을 통해 **가상의 답변(Hypothetical Document)** 을 먼저 만든 뒤,  
그 답변을 임베딩해 문서를 검색하는 기법입니다.

핵심 아이디어는 간단합니다.  
사용자 질문은 짧고 구어체인 경우가 많지만, 실제 문서는 길고 서술형이며 공식 용어를 쓰는 경우가 많습니다.  
HyDE는 이 간극을 줄이기 위해, 질문을 문서에 가까운 말투의 가상 답변으로 한 번 바꿔 검색에 활용합니다.

<img src="image/HyDE.png" width="550">

- 질문: 짧고, 의문문 형태이며, 추상적인 표현을 쓰기 쉬움
- 실제 문서: 길고, 서술형이며, 더 구체적이고 공식적인 표현을 사용함

이 표현 차이 때문에 질문 벡터가 정답 문서 근처에 잘 도달하지 못할 수 있습니다.  
HyDE는 LLM이 정답 문서와 비슷한 말투와 단어를 가진 가상의 답변을 만들게 해서, 이 간극을 줄이려는 접근입니다.


##### 예시: 사내 보안 규정 검색
원본 질문 : "카페에서 와이파이 잡아서 일해도 되나요?"
- 기존 검색의 한계: 실제 규정 문서에는 '카페', '와이파이' 같은 단어가 없을 수 있어 검색에 실패할 확률이 높습니다.

LLM이 생성한 가상 답변 (HyDE):
> "외부 장소에서 업무를 수행할 때는 공용 네트워크 보안 수칙에 따라 반드시 VPN을 연결해야 하며, 화면 보안 필름을 부착해야 합니다.  
인가되지 않은 네트워크 접속은 제한됩니다."

실제 정답 문서 (검색 타겟):  
> [제4조 원격 접속 보안] 사외망 접속 시 인가된 가상 사설망(VPN)을 경유해야 하며,  
개방형 무선 네트워크(Open SSID) 사용은 원칙적으로 금지한다.

##### HyDE의 효과와 한계

**효과:**
- 사용자가 정확한 도메인 용어를 몰라도 검색 성능을 개선할 수 있음
- 질문(Query)과 문서(Document) 간 표현 방식 차이를 LLM이 중간에서 보정
- 실제 문서와 유사한 말투·용어를 가진 가상 답변을 통해 벡터 유사도 상승
- 규정, 매뉴얼, 정책 문서처럼 **질문과 문서 스타일이 크게 다른 경우**에 특히 효과적

**주의사항**
- LLM이 해당 도메인에 대한 이해가 부족한 경우
  - 실제 문서와 무관한 키워드를 생성할 수 있음
  - **이 경우 오히려 검색 성능이 저하될 수 있음**
- 회사 고유 용어, 내부 은어, 프로젝트명 등은
  - HyDE 단독 사용보다 사전 용어 사전, 메타데이터, Hybrid Search와 병행하는 것이 안전함

따라서 HyDE는 **자연어 질문 ↔ 공식 문서 간 간극이 큰 환경에서 선택적으로 사용하는 보조 기법**으로 이해하는 것이 적절합니다.

##### HyDE 실습

In [ ]:
# 1. 안내서 스타일을 반영한 HyDE 프롬프트 정의
hyde_template = """
사용자의 질문에 대해, 아래 **[문서 성격]**을 바탕으로 짧은 설명문(가상의 답변)을 작성해 주세요.

[문서 성격]
- 금융투자협회의 투자자용 안내서
- 설명은 경고문, 유의사항, 대응 절차처럼 안내서 문체에 가깝게 작성
- 질문에 답하려면 중요할 법한 공식 용어와 행동 지침을 자연스럽게 포함

작성 규칙:
- 4~6줄 안쪽의 짧은 설명문으로 작성
- 질문에 바로 답하는 짧은 안내문처럼 쓸 것
- 문서에 있을 법한 표현과 절차 중심으로 쓸 것
- 질문과 무관한 다른 주제로 크게 확장하지 말 것
- 과도한 배경설명이나 일반 상식은 줄일 것

질문: {query}
가상 답변(문서 스타일 기반):
"""

hyde_prompt = PromptTemplate.from_template(hyde_template)

# 2. HyDE용 LLM 정의
hyde_llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# 3. HyDE 체인 구성
hyde_chain = hyde_prompt | hyde_llm | StrOutputParser()

# 4. 예시 질문 실행
hyde_query = "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?"
hypothetical_doc = hyde_chain.invoke({"query": hyde_query})
print("=== HyDE가 만든 가상 답변 ===")
print(hypothetical_doc)


In [ ]:
# 5. HyDE로 생성한 가상 문서를 임베딩해 관련 문서를 검색합니다.
results = vectorstore.similarity_search(hypothetical_doc, k=5)  # 가상 문서와 비슷한 실제 문서를 찾습니다.

print("=== [검색 결과] 실제 연결된 문서 내용 ===")
for i, r in enumerate(results[:3], 1):
    page = r.metadata.get("page")  # 원문 PDF의 페이지 정보를 꺼냅니다.
    page_text = f" / page {page + 1}" if isinstance(page, int) else ""  # 사람이 보는 페이지 번호로 바꿉니다.
    print(f"\n[문서 {i}{page_text}]")
    print(r.page_content[:300])  # 본문 앞부분만 확인해 연결 방향을 봅니다.


#### HyDE 사용 시 주의할 점

방금 실습처럼 HyDE가 관련 문서를 잘 끌어올릴 때도 있지만,
항상 원하는 방향으로 동작하는 것은 아닙니다.

가상 답변이 문서의 실제 표현과 어긋나면 검색 결과도 빗나갈 수 있습니다.

**왜 빗나갈 수 있나요?**

1. **문맥 불일치 (Context Mismatch)**  
   - **LLM의 상상:** "모바일 투자라면 앱 사용성, 편의성, 생산성 팁을 말하겠구나"
   - **실제 데이터:** "모바일폰 사용 시 유의사항", "비밀번호", "공인인증서", "보안", "금융사고"처럼 더 구체적이고 규정형인 표현이 등장

2. **키워드 오염 (Keyword Pollution)**  
   - LLM이 만든 가상 답변에 일반적인 디지털 금융 키워드가 너무 많으면,
     문서의 핵심 표현보다 일반적인 금융 문장이 우선 검색될 수 있습니다.

**교훈:**  
HyDE는 금융 안내서처럼 **정확한 용어와 절차**가 중요한 문서에서는
보조 기법으로만 써야 하며, 문서의 톤과 공식 용어를 최대한 반영해야 합니다.
        

### 2.3 Hybrid Retrieval

**Hybrid Retrieval**은 서로 다른 검색 방식의 약점을 서로 보완해, 정답 후보를 더 넓고 안정적으로 모으는 방식입니다.

| 방식 | 장점 | 단점 |
|---|---|---|
| 벡터 기반 의미 검색 | 표현이 달라도 의미가 비슷하면 잘 잡아냄 | 정확한 용어, 숫자, 법령 표기 검색에 약할 수 있음 |
| 키워드 기반 매칭 검색(BM25) | 용어 일치, 법령 구문, 코드, 표기 검색에 강함 | 표현이 조금만 달라도 쉽게 빗나갈 수 있음 |
| Hybrid | Dense가 놓친 정확한 용어 일치를 Sparse가 보완하고, Sparse가 놓친 의미 유사 표현을 Dense가 보완함 | 시스템 복잡성이 증가함 |

##### Hybrid 결과 병합 방식

Hybrid Retrieval에서는 Dense 검색과 BM25 검색을 각각 수행한 뒤, 두 결과를 어떤 방식으로 합칠지 결정해야 합니다.
아래 표는 검색 결과를 병합할 때 자주 쓰는 대표적인 방법들입니다.

| 방식 | 설명 | 특징 |
|---|---|---|
| **Parallel Retrieval** | Dense 검색과 BM25 검색을 각각 수행한 뒤 결과를 단순 병합 | 구현이 단순하지만 최종 순위가 불안정할 수 있음 |
| **Weighted Score Fusion** | 두 검색 방식의 점수를 정규화한 뒤 가중 평균으로 결합 | 가중치(alpha, beta)를 수동으로 조정해야 함 |
| **Reciprocal Rank Fusion (RRF)** | 각 검색 결과의 순위를 기반으로 점수를 계산해 결합 | 가중치 튜닝 없이도 안정적인 성능 |
| **EnsembleRetriever** | LangChain에서 제공하는 하이브리드 검색 조합기 | 내부적으로 여러 검색 전략을 조합 |

아래에서는 **RRF 기반 Hybrid Retrieval**을 중심으로 봅니다.



##### Hybrid Retrieval 실습



In [ ]:
from langchain_community.retrievers import BM25Retriever  # 키워드 기반 BM25 검색기를 만듭니다.

# Dense와 Sparse는 같은 split_docs 위에서 비교해야 Hybrid 결합 결과를 해석하기 쉽습니다.
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 임베딩 기반 검색기입니다.
bm25_retriever = BM25Retriever.from_documents(split_docs)  # 같은 청크를 BM25용 색인으로도 만듭니다.
bm25_retriever.k = 6  # BM25도 상위 6개까지 가져오게 맞춥니다.

# Hybrid 데모는 Sparse도 강점을 보일 수 있는 질문으로 통일합니다.
hybrid_demo_query = "금융투자회사 직원이 손실을 보전해 주겠다고 약속하면 왜 문제가 되는가?"

print("Dense/BM25 공통 청크 수:", len(split_docs))
print("Hybrid 비교 질문:", hybrid_demo_query)

#### Parallel Hybrid Retrieval (기본 조합)

Dense 검색과 Sparse 검색을 병렬로 수행한 뒤 결과를 단순히 이어 붙여 봅니다.



In [ ]:
# 실제로 테스트할 질문을 정합니다.
query = hybrid_demo_query  # Dense와 Sparse를 같은 질문으로 비교합니다.

dense_docs = dense_retriever.invoke(query)  # 의미 기반 검색 결과입니다.
sparse_docs = bm25_retriever.invoke(query)  # 키워드 기반 검색 결과입니다.

print(f"=== [Dense 검색 결과: {len(dense_docs)}개] ===")
for i, doc in enumerate(dense_docs[:4], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:140])  # 본문 앞부분만 봐도 검색 방향을 비교할 수 있습니다.
    print("---")

print(f"\n=== [Sparse(BM25) 검색 결과: {len(sparse_docs)}개] ===")
for i, doc in enumerate(sparse_docs[:4], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:140])  # 본문 앞부분만 봐도 검색 방향을 비교할 수 있습니다.
    print("---")

In [ ]:
# Dense 결과와 Sparse 결과를 그대로 이어 붙인 예시입니다.
# 이렇게 단순 합치면 중복이 생길 수 있고, 두 검색기의 순위도 그대로 섞여 들어갑니다.
combined_docs = dense_docs + sparse_docs

print(f"\n=== [Combined 결과: {len(combined_docs)}개] ===")
for i, doc in enumerate(combined_docs[:8], 1):
    print(f"[{i}] id={doc.metadata.get('id')}")
    print(doc.page_content[:110])
    print("---")

→ 단순 병합은 구현은 쉽지만, 결과에 **중복이 생길 수 있고** 두 검색기의 순위도 그대로 섞여 들어갑니다.  
즉, 같은 문서를 한 번만 남기고, 여러 검색기에서 **꾸준히 상위에 나온 문서**를 더 앞으로 올리는 정리가 필요합니다.


#### Reciprocal Rank Fusion (RRF)

RRF는 서로 다른 검색 결과의 **순위 정보만** 사용해 결과를 다시 합치는 방식입니다.

```
score = 1 / (k + rank + 1)
```

- `rank`: 각 검색 결과 안에서의 순위
- `k`: 순위 영향을 완만하게 만드는 상수. 보통 60을 많이 사용

핵심은 **한 검색기에서만 우연히 높게 나온 문서보다, 여러 검색기에서 꾸준히 상위에 등장한 문서**를 더 앞에 두는 것입니다.  
즉, Dense와 Sparse 둘 다에서 순위가 높은 문서가 위로 올라오도록 정리하는 방식이라고 이해하면 됩니다.


In [ ]:
from langchain_core.retrievers import BaseRetriever  # 커스텀 검색기를 LangChain 표준 Retriever처럼 쓰기 위해 상속합니다.

# 각 검색기의 순위만 사용해 문서별 누적 점수를 계산합니다.
def reciprocal_rank_fusion_with_scores(result_lists, k=60):  # 여러 검색기의 순위를 하나의 점수표로 합칩니다.
    """
    Reciprocal Rank Fusion(RRF)을 직접 구현한 함수

    result_lists : List[List[Document]]
        Dense, Sparse 등 여러 retriever가 반환한 검색 결과 리스트들의 묶음
        예: [dense_results, bm25_results]

    k : int
        순위 점수 차이를 너무 급격하게 만들지 않도록 완충하는 상수
        보통 60을 많이 사용합니다.

    반환값 : List[dict]
        {"doc": Document, "score": 누적점수} 형태의 리스트를 점수순으로 정렬해 돌려줍니다.
    """
    scores = {}  # 문서별 누적 점수를 저장합니다.

    for results in result_lists:  # Dense, Sparse처럼 각 검색기 결과를 차례로 봅니다.
        for rank, doc in enumerate(results):
            score = 1.0 / (k + rank + 1)  # 순위가 높을수록 더 큰 점수를 받습니다.
            doc_id = doc.metadata.get("id", doc.page_content)  # 같은 문서인지 비교할 기준 ID입니다.

            if doc_id not in scores:  # 처음 본 문서면 점수표에 등록합니다.
                scores[doc_id] = {"doc": doc, "score": 0.0}
            scores[doc_id]["score"] += score  # Dense 점수와 Sparse 점수를 문서별로 누적합니다.

    return sorted(scores.values(), key=lambda item: item["score"], reverse=True)

# 최종적으로는 점수순으로 정렬된 문서 리스트만 돌려줍니다.
def reciprocal_rank_fusion(result_lists, k=60):  # 점수표에서 문서 리스트만 꺼냅니다.
    """RRF 점수표에서 문서 객체만 꺼내 최종 순위 리스트로 반환합니다."""
    ranked = reciprocal_rank_fusion_with_scores(result_lists, k=k)  # 문서와 점수를 함께 계산합니다.
    return [item["doc"] for item in ranked]

# 여러 retriever를 한 번에 호출해 RRF로 합친 결과를 만듭니다.
def hybrid_rrf_retrieve(query, retrievers, k=60):
    """동일한 질문을 여러 retriever에 넣고, 결과를 RRF로 결합합니다."""
    results = [r.invoke(query) for r in retrievers]  # 같은 질문을 각 검색기에 넣습니다.
    return reciprocal_rank_fusion(results, k=k)

# LangChain 체인 안에서도 일반 retriever처럼 쓰기 위한 래퍼입니다.
class HybridRRFRetriever(BaseRetriever):
    retrievers: list  # 함께 결합할 검색기 목록입니다.
    k: int = 60  # RRF 점수 계산에 쓰는 완충 상수입니다.

    def _get_relevant_documents(self, query: str, *, run_manager=None):  # 일반 invoke에서 호출됩니다.
        return hybrid_rrf_retrieve(query, self.retrievers, k=self.k)

    async def _aget_relevant_documents(self, query: str, *, run_manager=None):  # 비동기 호출에서도 같은 로직을 씁니다.
        return hybrid_rrf_retrieve(query, self.retrievers, k=self.k)

# 출력에서 Dense 순위와 Sparse 순위를 함께 보기 위한 보조 맵입니다.
def build_rank_map(docs):  # 문서 ID를 보면 원래 몇 위였는지 바로 찾게 해 줍니다.
    return {doc.metadata.get("id", doc.page_content): rank + 1 for rank, doc in enumerate(docs)}

# 개별 검색 결과 확인 (비교용)
query = hybrid_demo_query

dense_results = dense_retriever.invoke(query)  # Dense 검색기의 원래 순위입니다.
bm25_results = bm25_retriever.invoke(query)  # Sparse 검색기의 원래 순위입니다.

# Hybrid retriever는 이후 Generation과 Evaluation에서도 그대로 재사용합니다.
hybrid_rrf = HybridRRFRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    k=60
)

# 같은 질문에 대해 RRF 점수표를 먼저 만들고, 그 순서대로 최종 문서를 봅니다.
rrf_ranked = reciprocal_rank_fusion_with_scores([dense_results, bm25_results], k=60)  # RRF 점수표를 만듭니다.
hybrid_results = [item["doc"] for item in rrf_ranked]  # 최종 문서 순위만 따로 꺼냅니다.

dense_rank_map = build_rank_map(dense_results)  # 문서별 Dense 순위를 빠르게 찾습니다.
sparse_rank_map = build_rank_map(bm25_results)  # 문서별 Sparse 순위를 빠르게 찾습니다.

print("=== RRF 상위 문서의 순위 비교 ===")
print("(참고: '-'는 해당 검색기의 top-k 결과에는 없었다는 뜻입니다.)")
for item in rrf_ranked[:5]:
    doc = item["doc"]
    doc_id = doc.metadata.get("id", doc.page_content)
    rrf_score = item["score"]
    print(
        f"id={doc_id} | dense_rank={dense_rank_map.get(doc_id, '-')} | "
        f"sparse_rank={sparse_rank_map.get(doc_id, '-')} | rrf_score={rrf_score:.4f}"
    )

print("\n=== RRF 상위 문서 본문 일부 ===")
for doc in hybrid_results[:3]:
    page = doc.metadata.get("page")
    page_text = f", page={page + 1}" if isinstance(page, int) else ""
    print("---")
    print(f"[id={doc.metadata.get('id')}{page_text}]")
    print(doc.page_content[:220])  # 본문 앞부분만 출력해 어떤 문서인지 확인합니다.


##### RRF의 장점
- 점수 스케일을 맞추지 않아도 됨
- Dense / Sparse 결과를 함께 쓸 때 안정적임
- 구현이 단순하면서도 실무에서 자주 사용됨



#### Hybrid Retrieval 결과 비교 실험

아래에서는 `단순 병합`과 달리, RRF가 **Dense와 Sparse 양쪽에서 모두 순위가 높게 나온 문서**를 위로 올리는지 봅니다.  
즉, 문서별로 `dense_rank`, `sparse_rank`, `rrf_score`를 함께 보면서 결과를 해석합니다.


In [ ]:
def preview_results(label, docs):  # 검색 결과 상위 2개를 짧게 보여 줍니다.
    print(f"\n=== {label} (Top 2) ===")
    for d in docs[:2]:
        print("---")
        print(d.page_content[:200])  # 본문 앞부분만 출력해 비교합니다.

# 실제로 테스트할 질문을 정합니다.
query = hybrid_demo_query  # 같은 질문으로 Dense, Sparse, Hybrid를 나란히 비교합니다.
#query = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"
#query = "부당권유란 무엇이며 어떤 경우 위법성이 문제되는가?"

dense_res = dense_retriever.invoke(query)  # Dense 검색 결과입니다.
sparse_res = bm25_retriever.invoke(query)  # BM25 검색 결과입니다.
hybrid_res = hybrid_rrf.invoke(query)  # 두 결과를 합친 Hybrid 검색 결과입니다.

preview_results("Dense Only", dense_res)
preview_results("Sparse Only", sparse_res)
preview_results("Hybrid (RRF, 양쪽에서 꾸준히 높은 문서 우선)", hybrid_res)

**관찰 포인트**

- 어떤 방식이 가장 관련성 높은 청크를 포함하는가?  
- Dense-only에서는 누락되지만 Sparse에서는 잡히는 내용이 있는가?  
- Hybrid가 두 방식의 장점을 잘 결합했는가?

#### 정리 — Retrieval 단계

- Dense, Sparse, Hybrid는 모두 **Retrieval 단계에서 후보를 모으는 전략**입니다.
- 이 단계의 핵심 목표는 **정답 가능성이 높은 문서를 빠뜨리지 않는 것**입니다.
- RRF도 Post-Retrieval이 아니라, **검색 결과를 합치는 Retrieval 단계의 fusion 기법**입니다.



### 2.4 후보군 점검과 Reranking

앞 절까지는 retrieval 단계에서 정답 후보를 넓게 모았습니다.  
이제는 이렇게 가져온 후보를 그대로 LLM에 넣지 않고, **어떤 후보를 남기고 어떻게 정리할지**를 보는 단계로 넘어갑니다.

이 구간은 **Post-Retrieval**에 해당합니다.  
즉, 검색이 끝난 뒤 후보 문서를 다시 읽고, 순서를 조정하거나, 불필요한 부분을 줄이거나, 중복을 제거하는 단계입니다.

여기서 볼 대표 기법은 다음과 같습니다.

| 기법 | 하는 일 |
|---|---|
| **Reranking** | 상위 후보를 다시 읽고 더 관련성 높은 순서로 재정렬 |
| **Filtering** | 관련성이 낮은 후보를 제거 |
| **Compression** | 문서 안에서 필요한 부분만 남김 |
| **Deduplication** | 비슷한 후보의 반복을 줄임 |

반대로 **RRF**는 이 단계의 기법이 아니라, retrieval 단계에서 여러 검색 결과를 합치는 **fusion 기법**입니다.

이 구간에서 먼저 확인할 것은 세 가지입니다.
- 상위 후보에 정답 단서가 들어왔는가
- 후보가 너무 길거나 중복되지 않는가
- 기본 retrieval만으로 충분한가, 아니면 reranking 같은 후처리가 필요한가

즉, 이 구간의 관심사는 **무엇을 찾을까**보다, **가져온 후보를 어떻게 더 쓰기 좋게 만들까**에 있습니다.


#### 2.4.1 Reranking

앞에서 retrieval 단계에서 Dense, Sparse, Hybrid로 후보를 먼저 모았습니다.  
이렇게 모은 상위 후보는 이미 어느 정도 관련 있지만, 질문과의 직접 관련도 순서가 완벽하지 않을 수 있습니다.

이럴 때 쓰는 단계가 **Reranking**입니다.  
즉, retrieval이 모아온 상위 후보를 대상으로 질문과 문서를 다시 비교해 순서를 더 정확하게 다듬습니다.

대표적으로 **cross-encoder**는 `query`와 `document`를 한 쌍으로 함께 읽고 관련도를 다시 점수화합니다.  
retrieval보다 느리기 때문에, 보통은 문서 전체가 아니라 **좁혀진 상위 후보 몇 개에만 선택적으로 적용**합니다.

실무에서는 이렇게 다시 매긴 점수를 기준으로 **상위 일부만 남겨 최종 컨텍스트 후보로 사용**하는 경우가 많습니다.  
즉, retrieval로 넓게 모은 뒤 reranking으로 순서를 다듬고, 그중 앞쪽 몇 개를 다음 단계로 넘기는 흐름입니다.

정리하면, retrieval은 **후보를 넓게 모으는 단계**, reranking은 **그 후보의 순서를 다시 정해 상위 일부를 추리는 단계**입니다.  
그다음에는 아래에서 보듯이, 남은 후보에서 중복을 줄이거나 필요한 부분만 남기는 **Filtering / Compression** 단계로 이어질 수 있습니다.


#### 2.4.2 Cross-Encoder Reranking 예시 코드

실무에서는 reranking에 **cross-encoder 계열 모델**을 많이 사용합니다.  
다만 이 실습 환경은 GPU가 없으므로, 아래 코드는 **실행용이 아니라 흐름을 보기 위한 참고 코드**로만 봅니다.

```python
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 가벼운 cross-encoder reranker를 고릅니다.
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 토크나이저와 reranker 모델을 준비합니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)
reranker = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
reranker.eval()

def cross_encoder_rerank(query, docs, top_k=None):
    # 질문과 각 문서를 한 쌍으로 묶습니다.
    text_pairs = [(query, d.page_content) for d in docs]

    # 모델 입력 형식으로 변환합니다.
    encoded = tokenizer(
        text_pairs,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    ).to(device)

    # 각 문서가 질문과 얼마나 잘 맞는지 점수를 계산합니다.
    with torch.no_grad():
        scores = reranker(**encoded).logits.squeeze(-1)

    scores = scores.cpu().numpy().tolist()

    # 점수가 높은 문서가 앞으로 오도록 다시 정렬합니다.
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    if top_k:
        ranked = ranked[:top_k]

    return ranked

# 예시: retrieval이 먼저 가져온 상위 10개 후보를 다시 정렬합니다.
query = "자연어 처리를 배우는 이유는 무엇인가?"
candidate_docs = dense_retriever.invoke(query)[:10]
reranked = cross_encoder_rerank(query, candidate_docs, top_k=5)

print("=== Cross-Encoder Reranking 결과 ===")
for doc, score in reranked:
    print(f"[Score: {score:.4f}] {doc.page_content[:200]}")
    print("---")
```


### 2.5 Context Filtering & Compression (Post-Retrieval)

#### 2.5.1 왜 Filtering & Compression이 중요한가?

LLM은 검색된 문서를 많이 넣는다고 항상 더 잘 답하지 않습니다.  
후보 문서를 그대로 넣으면 길이, 중복, 잡음 때문에 오히려 핵심 근거를 놓칠 수 있습니다.

| 문제 | 왜 생기나 |
|---|---|
| **컨텍스트가 너무 길어짐** | 컨텍스트 창에는 한계가 있고, 길어질수록 비용과 처리 부담도 커집니다. |
| **질문과 직접 관계없는 내용이 섞임** | 부록, 머리말, 반복 설명 같은 정보가 attention을 분산시켜 필요한 근거를 찾기 어려워집니다. |
| **관련 문서 안에서도 핵심 근거를 찾기 어려움** | 문서 전체는 관련 있어도, 답에 필요한 문장 몇 개만 흩어져 있으면 긴 문맥 속에서 놓치기 쉽습니다. |

Filtering과 Compression은 이런 문제를 줄여 **LLM이 실제 답변에 필요한 정보만 더 선명하게 보게 하는 과정**입니다.


#### 2.5.2 Context Filtering 기법

| 기법 | 무엇을 줄이는가 | 대표 방법 |
|---|---|---|
| **Relevance Filtering** | 질문과 직접 관련 없는 문서 | 점수 cutoff, LLM 기반 관련성 판별 |
| **Top-N / Score Cutoff** | 관련도 낮은 하위 후보 | reranker 기준 상위 N개만 유지, 점수 threshold 적용 |
| **Deduplication** | 같은 내용을 반복하는 문서 | 텍스트/임베딩 유사도 기반 제거 |

Reranking은 순서를 다시 정하는 단계이고, Filtering은 그 결과를 바탕으로 하위 후보를 버려 최종 컨텍스트를 좁히는 데 함께 쓰일 수 있습니다.

문서 안에서 필요한 부분만 남기는 방식은 아래 Compression 단계에서 함께 봅니다.


#### 실습: 검색 결과 중복 제거 (Deduplication)

아래 실습은 2.5.2 표에서 본 Deduplication을 직접 적용해 보는 예시입니다.

retrieval 이후에는 완전히 같은 청크가 아니어도, 비슷한 내용을 반복하는 청크가 여러 개 남을 수 있습니다.

Deduplication은 이런 유사 후보를 줄여, 검색 결과가 한 내용으로만 겹치지 않도록 정리하는 단계입니다.

핵심은 관련도를 다시 판단하는 것이 아니라, **100% 같은 중복만이 아니라 내용이 많이 겹치는 청크까지 함께 줄이는 것**입니다.


In [ ]:
import numpy as np  # 벡터 배열 계산과 유사도 계산에 쓰는 수치 연산 라이브러리입니다.
from langchain_openai import OpenAIEmbeddings  # 텍스트를 의미 기반 숫자 벡터로 바꿉니다.

# 검색 결과끼리 너무 비슷한 청크를 줄이기 위해 다시 임베딩합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# retrieval이 끝난 뒤, 내용이 거의 같은 청크를 제거합니다.
def dedupe_after_retrieval(docs, threshold=0.90):  # 내용이 거의 같은 청크를 줄입니다.
    if not docs:
        return [], []  # 비교할 문서가 없으면 바로 끝냅니다.

    texts = [d.page_content for d in docs]  # 중복 비교에 쓸 본문만 꺼냅니다.
    vectors = np.array(embeddings.embed_documents(texts))  # 각 문서를 임베딩 벡터로 바꿉니다.

    # 코사인 유사도를 쓰기 위해 벡터 길이를 1로 맞춥니다.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / (norms + 1e-12)

    keep_indices = []  # 남길 문서의 위치를 기록합니다.
    removed_indices = set()  # 중복으로 제거할 문서의 위치를 기록합니다.

    for i in range(len(docs)):
        if i in removed_indices:
            continue  # 이미 제거 대상으로 표시된 문서는 건너뜁니다.

        keep_indices.append(i)  # 현재 문서는 기준 문서로 남깁니다.
        sims = np.dot(vectors[i], vectors[i+1:].T)  # 현재 문서와 뒤쪽 문서의 유사도를 한 번에 계산합니다.
        for offset, sim in enumerate(sims, start=i+1):
            if sim >= threshold:
                removed_indices.add(offset)  # 너무 비슷하면 뒤쪽 문서를 제거 대상으로 표시합니다.

    kept_docs = [docs[i] for i in keep_indices]  # 최종적으로 남길 문서입니다.
    removed_docs = [docs[i] for i in sorted(removed_indices)]  # 중복으로 제거된 문서입니다.
    return kept_docs, removed_docs

# 같은 주제의 비슷한 규칙 청크가 많이 나오는 질문을 예시로 씁니다.
dedupe_query = "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?"
dense_candidates = dense_retriever.invoke(dedupe_query)[:4]  # Dense에서 상위 후보를 가져옵니다.
sparse_candidates = bm25_retriever.invoke(dedupe_query)[:4]  # Sparse에서 상위 후보를 가져옵니다.

# 어떤 검색기에서 온 문서인지 표시해 두면 제거 결과를 읽기 쉽습니다.
for doc in dense_candidates:
    doc.metadata["retrieval_source"] = "dense"
for doc in sparse_candidates:
    doc.metadata["retrieval_source"] = "bm25"

candidate_docs = dense_candidates + sparse_candidates  # 두 검색기의 후보를 한데 모읍니다.
deduped_docs, removed_docs = dedupe_after_retrieval(candidate_docs, threshold=0.88)  # 너무 비슷한 문서를 줄입니다.

print("중복 제거 전 개수:", len(candidate_docs))
print("중복 제거 후 개수:", len(deduped_docs))

print("\n=== 제거된 문서 ===")
for doc in removed_docs[:3]:
    print(f"[id={doc.metadata.get('id')}] source={doc.metadata.get('retrieval_source')}")
    print(doc.page_content[:160])
    print("---")

print("\n=== 중복 제거 후 남은 문서 ===")
for doc in deduped_docs[:5]:
    print(f"[id={doc.metadata.get('id')}] source={doc.metadata.get('retrieval_source', 'kept')}")
    print(doc.page_content[:160])
    print("---")

if not removed_docs:
    print("이번 질문에서는 제거된 문서가 많지 않지만, 비슷한 규칙 목록이 반복될 때는 차이가 더 커질 수 있습니다.")


#### 2.5.3 Context Compression 기법

Context Compression은 결국 **질문에 필요한 정보만 남기도록 문서를 줄이는 과정**이라고 이해하면 됩니다.
구현 방식은 크게 두 가지로 나눌 수 있습니다.

| 방식 | 무엇을 하느냐 |
|---|---|
| **Extractive Compression** | 원문에서 질문과 관련된 문장이나 구절만 뽑아 남김 |
| **Abstractive Compression** | 질문에 필요한 내용을 짧게 다시 표현해 새 문장으로 요약 |

즉, 둘 다 질문 기준으로 문맥을 줄이는 작업이고,
차이는 **원문을 그대로 남기느냐**, 아니면 **새 문장으로 다시 요약하느냐**에 있습니다.


#### 실습: 질문 기준 문맥 압축

아래 실습은 질문에 필요한 정보만 골라 새 문장으로 요약합니다.  
즉, 질문 기준으로 문맥을 줄이는 **요약형(Abstractive) Compression** 예시입니다.


In [ ]:
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.
from langchain_core.prompts import ChatPromptTemplate  # 여러 메시지 역할을 묶는 채팅 프롬프트 템플릿입니다.
from langchain_core.documents import Document  # 텍스트와 메타데이터를 함께 담는 문서 객체입니다.

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 질문 기준 문맥 압축에 사용할 생성 모델입니다.

# 질문 기준으로 필요한 내용만 새 문장으로 요약하는 프롬프트입니다.
compress_prompt = ChatPromptTemplate.from_template("""
당신은 '문서 요약 전문가'입니다.
주어진 [문서]에서 사용자의 [질문]에 답변하는 데 필요한 **핵심 정보**만 추출하여 요약하세요.

질문: {query}

문서:
{content}

출력 규칙:
- 질문과 직접 관련된 내용만 3~5줄로 요약
- 배경 설명, 문서 전체 요약은 제외
- 만약 문서에 질문과 관련된 내용이 전혀 없다면 "관련 정보 없음"이라고만 출력
""")

compress_chain = compress_prompt | llm  # Query-grounded 문맥 압축 체인입니다.

def compressed_retrieve(dense_retriever, query: str, k: int = 4):  # 검색 뒤 각 문서를 질문 기준으로 요약 압축합니다.
    dense_docs = dense_retriever.invoke(query)[:k]  # 먼저 상위 k개 문서를 가져옵니다.
    compressed_docs = []  # 요약된 문서를 다시 담아 둘 리스트입니다.

    for doc in dense_docs:  # 상위 문서를 하나씩 압축합니다.
        summary = compress_chain.invoke({"query": query, "content": doc.page_content}).content  # 질문에 필요한 내용을 새 문장으로 요약합니다.
        compressed_docs.append(Document(page_content=summary, metadata=doc.metadata))  # 요약 결과도 Document 형태로 맞춥니다.

    return compressed_docs

compression_query = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"
raw_docs = dense_retriever.invoke(compression_query)[:3]  # 압축 전 원본 문서를 봅니다.
compressed_docs = compressed_retrieve(dense_retriever, compression_query, k=3)  # 같은 문서를 질문 기준 압축 버전으로 만듭니다.

print("=== 압축 전 ===")
for i, doc in enumerate(raw_docs, 1):
    print(f"[{i}] id={doc.metadata.get('id')} / 길이={len(doc.page_content)}")
    print(doc.page_content[:180])  # 길이와 내용이 얼마나 줄었는지 비교합니다.
    print("---")

print("\n=== 압축 후 ===")
for i, doc in enumerate(compressed_docs, 1):
    print(f"[{i}] id={doc.metadata.get('id')} / 길이={len(doc.page_content)}")
    print(doc.page_content[:180])  # 길이와 내용이 얼마나 줄었는지 비교합니다.
    print("---")


**관찰 포인트**
- 검색된 문서보다 압축된 문서의 길이가 크게 줄어 있는가?  
- 질문과 직접 관련된 내용만 남고 주변 설명은 줄어들었는가?  
- 불필요한 서론/부록/표 등이 사라졌는가?

#### 정리

- **RRF**는 Dense와 Sparse 결과를 결합해 순서를 다시 정하는 retrieval 단계의 fusion입니다.
- **Filtering**은 불필요한 문서를 덜어내는 과정입니다.
- **Deduplication**은 Filtering 안에서 서로 비슷한 후보를 줄여 문맥 밀도를 높이는 방법입니다.
- **Compression**은 남길 문서 안에서 핵심만 짧게 만드는 과정입니다.


### 2.6 Generation 단계 최적화

검색이 잘 되어도, 답변을 어떻게 쓰게 하느냐에 따라 결과 품질은 크게 달라집니다.
핵심은 **프롬프트 구조**, **출처 표시**, **최종 답변 흐름**입니다.
        



#### 2.6.1 ~ 2.6.4 Generation 단계 요약

| 확인할 점 | 왜 중요한가 | 프롬프트에서 주로 하는 일 |
|---|---|---|
| 근거 제한 | 문서를 줘도 모델이 자기 말로 바꾸며 핵심을 놓칠 수 있음 | 문맥만 근거로 답하라고 명시 |
| 구역 구분 | 질문, 문맥, 출력 조건이 섞이면 규칙을 놓치기 쉬움 | 질문, 문맥, 출력 형식을 구역으로 나누기 |
| 환각 방지 | 검색 결과가 있어도 추측을 섞을 수 있음 | 문서에 없으면 없다고 답하게 하기 |
| 출처 표시 | 사용자가 답변 근거를 검증하기 쉬워지고 신뢰도가 높아짐 | 페이지 번호나 문서명과 페이지를 함께 적기 |
| 출력 형식 제어 | 답변 길이와 형식이 들쭉날쭉해질 수 있음 | 길이, 문장 수, 출처 표시 방식을 짧게 지정 |

즉, Generation 단계에서는 **무엇을 참고할지**, **어떻게 답할지**, **어떤 근거를 함께 보여줄지**를 프롬프트에서 함께 잡아주는 것이 중요합니다.


#### 실습: 근거 기반 답변을 위한 프롬프트 템플릿 구성

이 실습은 Generation 단계에서 **문서에 있는 내용만 근거로 답하게 하고**, 문서에 없는 내용을 추측해 말하는 일을 줄이기 위한 프롬프트를 만드는 과정입니다.  
즉, 검색된 문맥을 어떻게 넣고 어떤 규칙으로 답하게 할지를 템플릿 형태로 정리하는 실습입니다.

In [ ]:
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 호출합니다.

llm = ChatOpenAI(model="gpt-5-nano")  # 최종 답변 생성에 사용할 모델입니다.

# 검색된 문맥만 근거로 답하게 만드는 프롬프트입니다.
prompt = PromptTemplate.from_template("""
당신은 문서 기반 QA 시스템입니다.
아래 문맥(Context)에 기반해서만 답변하세요.
문맥에 없는 내용은 절대 생성하지 마십시오.

[질문]
{query}

[문맥]
{context}

[출력 형식]
- 문맥에 근거한 내용만 답한다.
- 문서에 없는 정보는 '문서에 해당 정보가 없습니다.'라고 답한다.

답변:
""")


#### 2.6.4 Citation / Source Attribution

문서 기반 답변에서는 **어느 문서를 보고 답했는지** 함께 보여주는 것이 좋습니다.
출처 표시는 답변 신뢰도를 높이고, 사용자가 검증하기 쉽게 만들어줍니다.

단일 PDF 실습에서는 `(p.38)`처럼 **페이지 번호**로 표시하는 방식이 가장 직관적입니다.
여러 문서를 함께 쓸 때는 필요하면 **문서명 + 페이지 번호**를 함께 적을 수 있습니다.


#### 실습: Citation 포함 최종 답변 생성

In [ ]:
# 출처를 함께 표시하도록 별도 프롬프트를 만듭니다.
citation_prompt = PromptTemplate.from_template("""
당신은 문서 기반 QA 시스템입니다.
아래 문맥(Context)에 기반해서만 답변하세요.
문맥에 없는 내용은 절대 생성하지 마십시오.

[질문]
{query}

[문맥]
{context}

[출력 형식]
- 문맥에 근거한 내용만 답한다.
- 각 문장은 가능한 한 관련 출처를 문장 끝에 (문서명, p.번호) 형태로 짧게 표시한다.
- 문서에 없는 정보는 '문서에 해당 정보가 없습니다.'라고 답한다.

답변:
""")

def format_context_with_sources(docs):  # 검색 문서를 출처 포함 문자열로 합칩니다.
    combined = ""  # 프롬프트에 넣을 최종 문맥 문자열입니다.
    for d in docs:
        source = d.metadata.get("source", "")
        source_name = source.split("/")[-1] if source else "문서"  # 파일명만 남겨 출처를 짧게 보여 줍니다.
        page = d.metadata.get("page")  # 페이지 번호가 있으면 함께 표기합니다.

        if isinstance(page, int):
            citation = f"{source_name}, p.{page + 1}"
        else:
            citation = source_name

        combined += f"[출처: {citation}]\n{d.page_content}\n\n"  # 문맥 안에서 바로 확인할 수 있는 출처 표기를 붙입니다.
    return combined

# 검색 -> 출처 포함 문맥 구성 -> 최종 답변 생성 흐름을 한 번에 실행합니다.
# 실제로 테스트할 질문을 정합니다.
query = "투자설명서와 약관은 왜 꼼꼼히 확인해야 하는가?"

docs = dense_retriever.invoke(query)  # 질문과 관련된 문서를 먼저 찾습니다.
formatted_context = format_context_with_sources(docs)  # 검색 결과를 출처 포함 문맥 문자열로 바꿉니다.

response = llm.invoke(citation_prompt.format(query=query, context=formatted_context))  # 출처 규칙을 포함한 프롬프트로 답변을 생성합니다.
print("=== 최종 답변 ===")
print(response.content)

#### 2.6.5 Structured Output / Fail-safe

실제 운영에서는 답변을 사람이 읽는 데서 끝나지 않고, **다른 시스템이 이어서 처리해야 하는 경우**가 많습니다.  
예를 들어 답변에 **문서 근거가 있는지**, **최종 답변이 무엇인지**, **출처가 어디인지**를 나눠 받아야 후속 화면이나 검증 로직에 연결하기 쉽습니다.

이럴 때는 프롬프트에 `JSON으로 답하라`고만 쓰기보다, **원하는 출력 구조를 스키마로 먼저 정해 두고** 그 형식에 맞게 응답받는 방식이 더 안정적입니다.
이런 방식을 보통 **Structured Output**이라고 부릅니다.

아래 예시는 `문서 근거 여부 / answer / source` 구조를 먼저 정한 뒤,
`has_evidence=False`이면 답변과 출처를 한 번 더 고정하는 **간단한 fail-safe**까지 붙인 형태입니다.

- `BaseModel`은 이런 **답변 양식의 설계도**라고 보면 됩니다.
- 코드에서는 `문서 근거 여부`를 `has_evidence`라는 이름으로 받습니다.
- Structured Output은 **출력 형태를 안정화**하는 역할을 합니다.
- 다만 값까지 완전히 보장되는 것은 아니므로, 필요한 경우 아래처럼 **후처리로 규칙을 한 번 더 고정**할 수 있습니다.


In [ ]:
from pydantic import BaseModel, Field  # 구조화된 출력 스키마를 정의합니다.

# 모델이 반환해야 할 답변 구조를 미리 정합니다.
class GuardedAnswer(BaseModel):
    has_evidence: bool = Field(description="문서 근거가 있는 답인지 여부")
    answer: str = Field(description="최종 답변")
    source: str = Field(description="관련 출처")

# 기존 llm에 구조화된 출력 스키마를 적용합니다.
structured_llm = llm.with_structured_output(GuardedAnswer)

# 답변 구조와 fallback 규칙을 함께 설명하는 프롬프트입니다.
guard_prompt = PromptTemplate.from_template("""
당신은 문서 기반 QA 시스템입니다.
아래 문맥(Context)에 기반해서만 답변하세요.
문맥에 없는 내용은 절대 생성하지 마십시오.

[질문]
{query}

[문맥]
{context}

문맥에 답이 있으면 has_evidence=true로 하고, answer에 답변을 작성하세요.
문맥에 답이 없으면 has_evidence=false로 하고, answer는 "문서에 해당 정보가 없습니다."로 작성하세요.
문맥에 답이 없으면 source는 빈 문자열로 두세요.
문맥에 답이 있으면 source에 문서명과 페이지를 짧게 적으세요.
""")

# 필요할 때는 고정 규칙을 한 번 더 적용해 출력 값을 안정화합니다.
def normalize_guarded_answer(response: GuardedAnswer) -> GuardedAnswer:
    if not response.has_evidence:
        return GuardedAnswer(
            has_evidence=False,
            answer="문서에 해당 정보가 없습니다.",
            source="",
        )
    return response

# 하나는 답이 있는 질문, 하나는 답이 없는 질문으로 테스트합니다.
test_queries = [
    "투자설명서와 약관은 왜 꼼꼼히 확인해야 하는가?",
    "이 안내서에 비트코인 ETF 투자 방법이 설명되어 있는가?",
]

for q in test_queries:
    docs = dense_retriever.invoke(q)[:3]  # 상위 문서 몇 개만 문맥으로 씁니다.
    context = format_context_with_sources(docs)  # 출처 포함 문맥으로 만듭니다.
    response = structured_llm.invoke(guard_prompt.format(query=q, context=context))  # 스키마에 맞는 구조화된 응답을 받습니다.
    response = normalize_guarded_answer(response)  # fail-safe 규칙을 한 번 더 적용합니다.

    result = {
        "has_evidence": response.has_evidence,
        "answer": response.answer,
        "source": response.source,
    }
    print(f"=== 질문: {q} ===")
    print(result)
    print()


확인할 점:

- `has_evidence / answer / source` 구조가 안정적으로 유지되는가?  
- 답이 없을 때 `has_evidence=False`와 고정 문구가 함께 반환되는가?  
- 문서 근거와 출처가 구조 안에서 함께 드러나는가?  

#### 2.6.6 정리

1. Generation 품질은 프롬프트 구조에 크게 좌우됩니다.
2. 문서 기반 답변에서는 Citation을 붙이는 습관이 중요합니다.
3. Structured Output과 Fail-safe는 운영 단계에서 응답 형식을 더 안정적으로 만드는 데 도움이 됩니다.
        

## 3. RAG 성능 평가 (Evaluation)

RAG는 단순히 돌아가는지보다 **잘 찾고, 근거 있게 답하는지**를 확인해야 개선이 가능합니다.

평가 범위는 두 가지로 둡니다.

1. **Retrieval 평가**: 상위 K개 안에 필요한 근거가 들어왔는가
2. **Generation 평가**: 답변이 문서에 근거했는가

<img src="image/eval.png" width="380">

Retrieval은 **Recall@K 중심**으로 보고,  
Generation은 **Faithfulness(문서 근거를 벗어나지 않았는가) 중심**으로 봅니다.


### 3.1 왜 RAG 평가가 중요한가?

다음과 같은 상황은 모두 **평가가 부실할 때** 흔히 발생하는 문제들입니다.

- 벡터 유사도 기반 검색만 사용하다 보니, 실제로 중요한 문서가 반복적으로 검색되지 않음  
- 검색은 잘 되었는데 LLM이 문서를 근거로 답하지 않고 자체 추론으로 대답함  
- Chunk 크기나 슬라이딩 윈도우 전략을 바꿔도 어떤 설정이 더 효과적인지 판단할 근거가 없음  
- Query Rewriting/Expansion을 적용했지만, 오히려 Recall이 떨어져 성능이 악화됨  
- BM25·Vector·Hybrid Retrieval 실험을 해도 정량적 지표 없이 “감”으로 방식이 선택됨  

따라서 **RAG 파이프라인을 안정적으로 개선하기 위해서는 체계적 평가가 필수적입니다.**

### 3.2 Retrieval Evaluation (Hit@K / Recall@K 중심)

Retrieval 평가는 크게 두 가지를 봅니다.  
하나는 **정답 근거를 최소 1개라도 놓치지 않았는지**, 다른 하나는 **관련 청크를 얼마나 폭넓게 회수했는지**입니다.

| 지표 | 정의 | 언제 보면 좋은가 |
|---|---|---|
| **Hit@K** | 상위 K개 안에 수동으로 지정한 관련 청크가 **하나라도** 들어왔는가 | 정답 근거를 최소 1개라도 찾았는지 빠르게 확인할 때 |
| **Recall@K** | **수동으로 지정한 관련 청크 전체** 중에서, 상위 K개 안에 **몇 개가 들어왔는가** | 필요한 정보를 얼마나 빠짐없이 회수했는지 볼 때 |
| **Precision@K** | **상위 K개 결과** 중에서, 수동으로 지정한 관련 청크가 **몇 개 들어왔는가** | 상위 결과에 잡음이 많은지 볼 때 |
| **MRR / nDCG** | 관련 근거가 얼마나 앞순위에 배치되었는가 | 순위 품질까지 함께 보고 싶을 때 |

여기서 `관련 청크`는 retriever가 가져온 결과가 아니라, 질문별로 **미리 수동 확인해 둔 정답 청크**를 뜻합니다.
즉, Recall은 **정답 전체 기준으로 얼마나 회수했는지**, Precision은 **가져온 결과 기준으로 얼마나 정확한지**를 보는 지표입니다.

이 자료에서는 표준 지표인 **Hit@K**와 **Recall@K**를 사용합니다.
- **Hit@K**: 상위 K개 안에 관련 청크가 하나라도 포함되었는지 평가
- **Recall@K**: 수동으로 지정한 관련 청크가 상위 K개 안에 얼마나 포함되었는지 평가

> 참고: 질문마다 관련 청크를 1개만 두면 Recall@K가 Hit@K와 같아질 수 있습니다.  
가능하면 질문별로 관련 청크를 2개 이상 지정해 두면 Recall@K를 더 분명하게 해석할 수 있습니다.


#### Retrieval 평가용 데이터 구조

`test_cases`에는 질문과 관련 청크 ID를 함께 둡니다.

- `relevant_doc_ids`: 수동 확인한 관련 청크 ID



#### 실습: Retrieval 지표 계산



In [ ]:
# 평가에서는 모든 검색 결과가 아니라 상위 k개만 잘 가져왔는지 보는 경우가 많습니다.
def retrieve_docs(retriever, query, k=5):
    return retriever.invoke(query)[:k]  # 검색 결과 중 상위 k개만 평가에 사용합니다.

#### 수동 평가셋 정의

아래 `test_cases`는 `split_docs`를 확인해 고른 수동 평가셋입니다. 

In [ ]:
# split_docs를 확인해 고른 수동 평가셋입니다.
# - query: 실제 사용자가 할 법한 질문
# - relevant_doc_ids: 새 청킹 기준에서 확인한 관련 청크 ID

test_cases = [
    {
        "query": "왜 금융투자상품 투자는 반드시 본인 명의로 해야 하는가?",
        "relevant_doc_ids": [36],
    },
    {
        "query": "투자설명서와 약관을 받아 두고 보관해야 하는 이유는 무엇인가?",
        "relevant_doc_ids": [38],
    },
    {
        "query": "가족이나 지인에게 계좌번호와 비밀번호를 알려주면 왜 위험한가?",
        "relevant_doc_ids": [39],
    },
    {
        "query": "전산장애가 있었을 때 매도의사를 입증하려면 어떤 자료를 남겨야 하는가?",
        "relevant_doc_ids": [444, 446],
    },
    {
        "query": "모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?",
        "relevant_doc_ids": [63],
    },
    {
        "query": "직원에게 계좌관리를 맡기기로 했다면 계약서에 어떤 내용을 구체적으로 적어야 하는가?",
        "relevant_doc_ids": [101],
    },
    {
        "query": "내가 맡긴 적 없는데 직원이 알아서 거래했다면 어떤 유형의 분쟁으로 볼 수 있는가?",
        "relevant_doc_ids": [126],
    },
    {
        "query": "직원에게 투자판단을 맡긴 계약은 법적으로 어떤 성격으로 이해해야 하는가?",
        "relevant_doc_ids": [162],
    },
    {
        "query": "경험이 부족한 투자자에게 어떤 식으로 권하면 부당권유 문제가 생길 수 있는가?",
        "relevant_doc_ids": [198],
    },
    {
        "query": "불완전판매는 왜 부당권유의 한 유형으로 함께 다뤄지는가?",
        "relevant_doc_ids": [125],
    },
    {
        "query": "주문이 다르게 체결되거나 누락됐다면 회사는 무엇을 확인했어야 하는가?",
        "relevant_doc_ids": [375, 376],
    },
    {
        "query": "보이스피싱 피해를 입었다면 어떤 순서로 대응해야 하는가?",
        "relevant_doc_ids": [469],
    },
    {
        "query": "유사수신 업체는 어떤 특징을 보이며 왜 위험한가?",
        "relevant_doc_ids": [456],
    },
    {
        "query": "회사와 먼저 직접 분쟁을 풀어보는 방식이 왜 가장 손쉽고 합리적일 수 있는가?",
        "relevant_doc_ids": [108, 109],
    },
    {
        "query": "금융투자회사 직원이 손실을 보전해 주겠다고 약속하면 왜 문제가 되는가?",
        "relevant_doc_ids": [13, 543],
    },
]

test_cases[:5]  # 평가셋 구조가 잘 들어갔는지 앞부분만 확인합니다.


#### Hit@K / Recall@K 계산



In [ ]:
# 검색 결과에서 문서 id만 뽑아 비교하기 쉽게 만듭니다.
def retrieved_ids(docs):  # 검색 결과를 ID 목록으로 바꿉니다.
    return [doc.metadata.get("id") for doc in docs]  # 검색된 문서들의 id만 뽑습니다.

# 상위 K개 안에 관련 청크가 하나라도 들어왔는지 계산합니다.
def hit_at_k(retrieved_docs, relevant_doc_ids):
    pred_ids = {doc.metadata.get("id") for doc in retrieved_docs}  # 상위 K개의 문서 ID 집합입니다.
    return bool(pred_ids & set(relevant_doc_ids))

# 수동으로 지정한 관련 청크를 상위 K개가 얼마나 회수했는지 계산합니다.
def recall_at_k(retrieved_docs, relevant_doc_ids):
    pred_ids = {doc.metadata.get("id") for doc in retrieved_docs}  # 상위 K개의 문서 ID 집합입니다.
    matched_ids = [doc_id for doc_id in relevant_doc_ids if doc_id in pred_ids]  # 실제로 회수한 관련 청크만 남깁니다.
    if not relevant_doc_ids:
        return 0.0, []
    return len(matched_ids) / len(relevant_doc_ids), matched_ids

# 질문별 Hit@K 평균을 구합니다.
def mean_hit_at_k(retriever, test_cases, k=5):
    correct = 0  # Hit한 질문 수를 셉니다.
    total = len(test_cases)

    for case in test_cases:
        retrieved_docs = retrieve_docs(retriever, case["query"], k=k)  # 각 질문의 top-k 결과를 가져옵니다.
        if hit_at_k(retrieved_docs, case["relevant_doc_ids"]):
            correct += 1

    return correct / total

# 질문별 Recall@K 평균을 구합니다.
def mean_recall_at_k(retriever, test_cases, k=5):
    scores = []  # 질문별 Recall@K 값을 모읍니다.
    for case in test_cases:
        retrieved_docs = retrieve_docs(retriever, case["query"], k=k)  # 각 질문의 top-k 결과를 가져옵니다.
        recall, _ = recall_at_k(retrieved_docs, case["relevant_doc_ids"])  # 관련 청크를 얼마나 찾았는지 계산합니다.
        scores.append(recall)
    return sum(scores) / len(scores)


#### Hybrid Retrieval 상세 로그 보기



In [ ]:
def debug_retrieval_metrics(retriever, test_cases, k=5):  # 질문별 Hit@K와 Recall@K를 자세히 출력합니다.
    for case in test_cases:
        query = case["query"]
        relevant_ids = case["relevant_doc_ids"]
        retrieved_docs = retrieve_docs(retriever, query, k=k)
        pred_ids = retrieved_ids(retrieved_docs)  # 출력용으로 top-k ID를 정리합니다.
        recall, matched_ids = recall_at_k(retrieved_docs, relevant_ids)  # 관련 청크를 얼마나 찾았는지 계산합니다.
        hit = hit_at_k(retrieved_docs, relevant_ids)  # 하나라도 찾았는지 계산합니다.

        print(f"\n[질문] {query}")
        print(f"- 예측 Top-{k} IDs: {pred_ids}")
        print(f"- 수동 확인한 관련 IDs: {relevant_ids}")
        print(f"- Hit@{k}: {hit}")
        print(f"- Recall@{k}: {recall:.2f}")
        print(f"- 회수된 관련 IDs: {matched_ids}")
        print("-" * 50)

print("=== Hybrid Retrieval 상세 로그 ===")
debug_retrieval_metrics(hybrid_rrf, test_cases, k=5)

In [ ]:
# 검색기별로 Hit@K와 Recall@K를 한 번에 비교합니다.
print("Dense Hit@5:", mean_hit_at_k(dense_retriever, test_cases, k=5))
print("Sparse Hit@5:", mean_hit_at_k(bm25_retriever, test_cases, k=5))
print("Hybrid Hit@5:", mean_hit_at_k(hybrid_rrf, test_cases, k=5))

print()

print("Dense Recall@5:", mean_recall_at_k(dense_retriever, test_cases, k=5))
print("Sparse Recall@5:", mean_recall_at_k(bm25_retriever, test_cases, k=5))
print("Hybrid Recall@5:", mean_recall_at_k(hybrid_rrf, test_cases, k=5))

print()
print("권장 해석: Hit@K는 관련 청크를 하나라도 찾았는지 보여주고,")
print("Recall@K는 수동으로 지정한 관련 청크를 얼마나 빠짐없이 회수했는지 보여줍니다.")


**관찰 포인트**

- **Hit@K**가 높을수록 관련 청크를 하나라도 놓치지 않았다고 볼 수 있습니다.
- **Recall@K**가 높을수록 수동으로 지정한 관련 청크를 더 넓게 회수했다고 해석할 수 있습니다.
- 두 지표가 함께 높으면 관련 근거를 안정적으로 회수한 경우에 가깝습니다.
- 두 지표가 어긋나면 청크 경계, 질문 표현, 관련 청크 라벨링 범위를 함께 점검하는 것이 좋습니다.


### 3.3 Generation Evaluation (Faithfulness 중심)

여기서 **Faithfulness**는 답변이 문서 근거를 벗어나지 않았는지,  
즉 **문서에 없는 내용을 덧붙이지 않았는지**를 보는 지표입니다.

생성 평가는 여러 축으로 볼 수 있지만, 먼저 Faithfulness를 확인합니다.  
문서에 없는 말을 하지 않았는지 보는 것이 가장 기본적인 안전장치이기 때문입니다.

참고로 이후에는 아래 항목도 함께 확장해서 볼 수 있습니다.

| 평가 항목 | 무엇을 보는가 |
|---|---|
| **Faithfulness** | 답변이 문서 근거를 벗어나지 않았는가. 즉 문서에 없는 내용을 추측으로 덧붙이지 않았는가 |
| **Relevance** | 답변이 질문 의도에 맞는가. 문서 내용이 맞더라도 엉뚱한 방향으로 답하지 않았는가 |
| **Completeness** | 질문에 답하는 데 필요한 핵심 정보를 빠뜨리지 않았는가. 맞는 말만 했지만 중요한 부분이 비지 않았는가 |
| **Citation Quality** | 출처 표시가 실제 답변 내용과 잘 연결되는가. 문서명·페이지가 자연스럽고 검증 가능하게 붙어 있는가 |


#### Generation Evaluation 실습: LLM을 활용한 Faithfulness 평가

`LLM-as-a-Judge`는 다른 LLM에게 답변 품질을 평가하게 하는 방식입니다.
사람이 모든 답변을 직접 읽기 어려울 때, **1차 자동 점검기**로 활용하기 좋습니다.


In [ ]:
judge_prompt = """
다음은 RAG 모델이 생성한 답변과, 참조 문서(Context)입니다.
답변이 문서에 근거했는지 0~1 점수로 평가하세요.

- 1: 문서에서 근거를 명확히 가지고 있음
- 0: 문서에 없는 내용을 생성함 (hallucination)
- 0.5: 부분적으로만 근거가 있음

[질문]
{query}

[답변]
{answer}

[문맥]
{context}

0, 0.5, 1 중 하나의 숫자만 출력하세요.
"""

from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
# 문서 근거 여부만 따로 평가하기 위한 프롬프트입니다.
judge_template = PromptTemplate.from_template(judge_prompt)  # 평가용 문자열을 재사용 가능한 템플릿으로 바꿉니다.

def evaluate_answer(query, answer, context):  # 답변 하나의 faithfulness 점수를 계산합니다.
    judge_llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 평가도 같은 소형 모델로 진행합니다.
    judge_chain = judge_template | judge_llm | StrOutputParser()  # 평가 프롬프트와 평가 모델을 하나의 체인으로 묶습니다.
    return judge_chain.invoke({"query": query, "answer": answer, "context": context})  # 답변의 faithfulness 점수를 계산합니다.


In [ ]:
# 실제로 테스트할 질문을 정합니다.
query = "금융투자상품 투자는 누구의 명의로 해야 하는가?"

docs = hybrid_rrf.invoke(query)  # Hybrid 검색으로 관련 문서를 가져옵니다.
context = format_context_with_sources(docs)  # 검색 결과를 평가용 문맥으로 합칩니다.

answer = llm.invoke(prompt.format(query=query, context=context)).content  # 문맥 기반 답변을 생성합니다.

score = evaluate_answer(query, answer, context)  # 생성 답변이 문서에 근거했는지 채점합니다.

print("=== 답변 ===")
print(answer)

print("\n=== Faithfulness Score ===")
print(score)


## 4. RAG 운영 및 모니터링 (Monitoring & Observability)

### 4.1 왜 RAG에는 모니터링이 필요한가?

RAG는 한 번의 모델 호출이 아니라 **검색 -> 문맥 구성 -> 생성**으로 이어지는 파이프라인입니다.
그래서 답변이 어긋났을 때도 원인이 한 군데가 아닐 수 있습니다.

- 검색된 문서가 질문과 맞지 않았는가?
- 문서는 맞지만 context가 너무 길거나 불필요한 정보가 많았는가?
- 프롬프트가 문맥을 제대로 활용하지 못했는가?
- 생성 단계에서 문서에 없는 말을 보탰는가?

이런 문제는 **최종 답변만 보고는 구분하기 어렵습니다.**
그래서 RAG에서는 결과 평가와 별개로, **과정 전체를 관찰하는 모니터링**이 필요합니다.

<img src="image/RAG.png" width="600">


### 4.2 RAG Observability 개념과 도구 선택

RAG에서 Observability는 **한 번의 요청이 어떤 과정을 거쳐 최종 답변에 도달했는지 단계별로 볼 수 있는 상태**를 뜻합니다.  
최종 답변만 보는 것이 아니라, 중간 과정까지 함께 보여야 원인을 찾을 수 있습니다.

RAG 관점에서는 특히 아래 항목이 보여야 합니다.

- 사용자가 어떤 질문을 입력했는가?
- Retriever가 어떤 문서를 반환했는가?
- 실제로 LLM에 전달된 context와 prompt는 무엇인가?
- 각 단계에 얼마나 시간이 걸렸는가?
- 최종 답변이 어떤 근거를 바탕으로 생성되었는가?

좋은 Observability 도구는 이런 정보를 **한 요청 단위의 trace**로 묶어 보여줍니다.  
대표적으로 **LangSmith**와 **Langfuse**가 많이 쓰이며, 둘 다 trace를 남길 수 있지만 운영 방식과 활용 환경에는 차이가 있습니다.

다음 절에서는 두 도구를 비교하고, 이 자료에서는 **Langfuse**를 기준으로 실습합니다.


### 4.3 LangSmith vs Langfuse 비교

두 도구의 공통 목적은 같습니다.

- RAG 요청 단위 추적
- Prompt / Context / Output 기록
- 단계별 디버깅과 성능 분석 지원

차이는 아래처럼 정리할 수 있습니다.

| 구분 | LangSmith | Langfuse |
|---|---|---|
| 개발 주체 | LangChain | Langfuse |
| 프레임워크 의존성 | LangChain 중심 | 프레임워크 독립 |
| 배포 방식 | 관리형 중심 | 오픈소스 / 자체 호스팅 가능 |
| 기업 환경 적합성 | 빠른 실험용 | 내부 시스템 연동에 유리 |

**실무 환경에서도 자주 쓰이는 Langfuse**를 기준으로 trace를 남겨봅니다.


### 4.4 Langfuse 시작하기

#### 1) Langfuse 회원가입 및 Organization 생성

- https://langfuse.com 에 접속합니다.
- 회원가입 후 로그인합니다.
- Organization을 생성합니다.

Langfuse는 프로젝트를 **조직 단위로 관리**하므로, 먼저 Organization을 만들어야 합니다.

<img src="image/langfuse_organization.png" width="650">

<br>

<img src="image/langfuse_organization2.png" width="650">


#### 2) 프로젝트(Project) 생성

- 생성한 Organization 안에서 새 프로젝트를 만듭니다.
- 프로젝트 이름과 기본 설정을 입력합니다.

실습용 프로젝트와 운영용 프로젝트를 나눠 두면 trace 관리가 더 편합니다.

<img src="image/langfuse_project.png" width="650">


#### 3) API Key 발급

- Project Settings로 이동합니다.
- `Public Key`와 `Secret Key`를 확인합니다.

`Public Key`는 프로젝트 식별용, `Secret Key`는 서버 인증용으로 생각하면 이해하기 쉽습니다.

<img src="image/langfuse_api_key.png" width="650">

<br>

<img src="image/langfuse_api_key2.png" width="650">


#### 4) .env 파일에 환경 변수 설정

Langfuse Cloud를 사용할 때는 아래처럼 설정하면 됩니다.

```bash
LANGFUSE_PUBLIC_KEY=...
LANGFUSE_SECRET_KEY=...
LANGFUSE_HOST=https://cloud.langfuse.com
```

자체 호스팅 환경이라면 `LANGFUSE_HOST`를 사내 Langfuse 주소로 바꾸면 됩니다.
`.env` 파일은 API Key와 마찬가지로 외부 저장소에 올리지 않도록 주의합니다.


### 4.5 실습: Langfuse CallbackHandler로 RAG 요청 추적하기

Langfuse의 `CallbackHandler`는 **LangChain 실행 흐름을 trace로 남기는 연결 지점**입니다.
이 핸들러를 `invoke()`의 `config`에 넣으면 아래 단계가 하나의 trace로 이어집니다.

- Retriever 검색
- Context 구성
- Prompt 적용
- LLM 호출
- 최종 Output 생성

#### 1) Langfuse CallbackHandler 생성


In [ ]:
from dotenv import load_dotenv  # .env 안의 LANGFUSE_* 값을 현재 세션으로 불러옵니다.
from langfuse.langchain import CallbackHandler  # LangChain 실행 기록을 Langfuse trace로 보냅니다.

load_dotenv(override=True)  # Langfuse 설정을 현재 노트북 세션에 반영합니다.
langfuse_handler = CallbackHandler()  # 이후 invoke에서 사용할 추적 핸들러를 만듭니다.

#### 2) 기존 RAG 체인 실행에 CallbackHandler 전달

앞에서 이미 만든 `hybrid_rrf`, `prompt`, `llm`, `format_context_with_sources`를 그대로 사용합니다.
핵심은 `config={"callbacks": [langfuse_handler]}` 한 줄을 추가하는 것입니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼냅니다.
from langchain_core.runnables import RunnableLambda, RunnablePassthrough  # 기존 함수와 retriever를 LCEL 체인으로 묶습니다.

rag_chain = (
    {
        "query": RunnablePassthrough(),  # 사용자 질문은 그대로 프롬프트의 query로 전달합니다.
        # hybrid_rrf는 BaseRetriever를 상속했으므로 LCEL 파이프에 바로 연결할 수 있습니다.
        "context": hybrid_rrf | RunnableLambda(format_context_with_sources),  # 검색 결과를 하나의 문맥 문자열로 바꿉니다.
    }
    | prompt  # 질문과 문맥을 답변용 프롬프트로 구성합니다.
    | llm  # 문맥 기반 답변을 생성합니다.
    | StrOutputParser()  # 응답 객체에서 텍스트만 추출합니다.
)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

answer = rag_chain.invoke(
    question,
    config={"callbacks": [langfuse_handler]},  # 이 설정이 trace를 Langfuse에 남깁니다.
)

print(answer)


실행 후 Langfuse 화면에서는 **입력 질문 -> 검색 결과 -> 전달된 context -> 최종 응답** 흐름이 순서대로 보입니다.  
즉, 결과가 이상할 때 "검색이 문제인지, 생성이 문제인지"를 훨씬 빨리 좁혀갈 수 있습니다.

<img src="image/langfuse_tracing.png" width="800">

같은 방식으로 `ainvoke()`, `batch()`, `stream()`에도 callback config를 넘기면 실행 방식이 달라도 일관되게 trace를 남길 수 있습니다.


### 4.6 Langfuse 주요 기능 살펴보기

Langfuse는 단순히 한 번의 실행 로그만 보는 도구가 아닙니다.
RAG 품질을 꾸준히 개선하기 위해 필요한 관측 기능을 함께 제공합니다.

<table style="border-collapse:collapse; width:100%; text-align:left;">
  <thead>
    <tr>
      <th style="border:1px solid #ccc; padding:8px; width:18%;">영역</th>
      <th style="border:1px solid #ccc; padding:8px; width:22%;">기능</th>
      <th style="border:1px solid #ccc; padding:8px; width:30%;">설명</th>
      <th style="border:1px solid #ccc; padding:8px;">활용 목적</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="2" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Tracing</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">Trace</td>
      <td style="border:1px solid #ccc; padding:8px;">하나의 사용자 요청 단위 실행 흐름</td>
      <td style="border:1px solid #ccc; padding:8px;">요청 단위 디버깅, 성능 비교</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">Observation</td>
      <td style="border:1px solid #ccc; padding:8px;">Trace 내부의 개별 단계</td>
      <td style="border:1px solid #ccc; padding:8px;">단계별 병목 분석</td>
    </tr>
    <tr>
      <td rowspan="2" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Sessions / Users</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">Sessions</td>
      <td style="border:1px solid #ccc; padding:8px;">여러 요청을 하나의 흐름으로 관리</td>
      <td style="border:1px solid #ccc; padding:8px;">사용자 여정 분석</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">Users</td>
      <td style="border:1px solid #ccc; padding:8px;">사용자 단위 실행 기록 관리</td>
      <td style="border:1px solid #ccc; padding:8px;">사용자별 패턴 파악</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Prompt Management</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">Prompts</td>
      <td style="border:1px solid #ccc; padding:8px;">프롬프트를 자산처럼 관리</td>
      <td style="border:1px solid #ccc; padding:8px;">버전 관리, 실험 비교</td>
    </tr>
    <tr>
      <td rowspan="4" style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Evaluation</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">Scores</td>
      <td style="border:1px solid #ccc; padding:8px;">출력 결과에 점수 부여</td>
      <td style="border:1px solid #ccc; padding:8px;">품질 정량 평가</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">LLM-as-a-Judge</td>
      <td style="border:1px solid #ccc; padding:8px;">다른 LLM을 평가자로 사용</td>
      <td style="border:1px solid #ccc; padding:8px;">자동 평가</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">Human Annotation</td>
      <td style="border:1px solid #ccc; padding:8px;">사람이 직접 평가</td>
      <td style="border:1px solid #ccc; padding:8px;">정성 평가</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px;">Datasets</td>
      <td style="border:1px solid #ccc; padding:8px;">질문/정답 데이터셋 관리</td>
      <td style="border:1px solid #ccc; padding:8px;">반복 실험, 회귀 테스트</td>
    </tr>
    <tr>
      <td style="border:1px solid #ccc; padding:8px; vertical-align:top;"><strong>Dashboards</strong></td>
      <td style="border:1px solid #ccc; padding:8px;">Dashboards</td>
      <td style="border:1px solid #ccc; padding:8px;">주요 지표 시각화</td>
      <td style="border:1px solid #ccc; padding:8px;">운영 현황 모니터링</td>
    </tr>
  </tbody>
</table>


<img src="image/langfuse_main.png" width="800">

### 4.7 Langfuse를 활용한 품질 분석 사례

Langfuse에 trace가 쌓이면 아래와 같은 분석이 쉬워집니다.

- 답변이 부정확한 요청에서 Retriever 결과가 적절했는지 확인
- context가 너무 길거나 중복 문서가 많이 들어갔는지 확인
- 같은 질문에 대해 프롬프트 버전 차이가 어떤 영향을 주는지 비교
- 품질 저하 원인이 Retrieval인지 Generation인지 구분


### 4.8 평가(Evaluation)와 모니터링의 역할 차이

Evaluation과 Monitoring은 목적이 다릅니다.

- **Evaluation**: 결과가 좋은지 점수나 기준으로 판단
- **Monitoring**: 왜 이런 결과가 나왔는지 과정 중심으로 분석

실무에서는 보통 **평가로 품질을 재고, 모니터링으로 원인을 찾습니다.**


### 4.9 정리

- RAG는 한 번 구축하고 끝나는 시스템이 아니라, 운영 중 계속 관찰하고 개선해야 하는 시스템입니다.
- Langfuse를 쓰면 Retrieval -> Context -> Generation 흐름을 요청 단위로 확인할 수 있습니다.
- Trace를 꾸준히 보면 병목과 실패 패턴을 더 빨리 찾을 수 있습니다.


## 5. 최종 과제 안내

최종 과제는 별도 노트북 **`최종과제.ipynb`** 에서 진행합니다.



### 5.1 권장 과제 방식

최종 과제는 별도 노트북의 **RAG 점수 개선 과제**로 수행합니다.

- 수료 기준: 채점 결과 **15점 이상**
- 제출물: **점수 캡처 이미지 + `수행평가 제출양식_이름.docx` 1개**



### 5.2 진행 방법

진행 순서는 아래와 같습니다.

1. `최종과제.ipynb`의 baseline을 실행합니다.
2. Retrieval / Post-Retrieval / Generation 중 개선 포인트를 2개 이상 적용합니다.
3. 개선 전/후 점수를 비교합니다.
4. 최종 점수 화면을 캡처합니다.
5. `수행평가 제출양식_이름.docx`에 시도한 방법을 짧게 정리합니다.



## 6. 참고자료

| 기법 | 대표 참고 논문 | 메모 |
|---|---|---|
| Multi-Query / Query Expansion | Wang et al., *Query2doc: Query Expansion with Large Language Models* (EMNLP 2023) | LLM 기반 질의 확장 흐름 참고 |
| HyDE | Gao et al., *Precise Zero-Shot Dense Retrieval without Relevance Labels* (ACL 2023) | 가상 문서 기반 dense retrieval 참고 |
| RRF | Cormack et al., *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods* (SIGIR 2009) | 여러 검색 결과의 순위 기반 결합 참고 |
| Cross-encoder Reranking | Nogueira and Cho, *Passage Re-ranking with BERT* (2019) | query-document 쌍 재점수화 예시 |
